In [1]:
import pandas as pd

import src
from src.load import DataLoader

In [2]:
pd.set_option("display.max_rows", 1024)
pd.set_option("display.max_colwidth", 256)

In [3]:
dl = DataLoader()

In [4]:
videos = (
    dl.videos(filtered=True)
    .join(dl.channels(), "channel_id")
    .select(
        ["video_id", "channel", "video_likes", "video_views", "video_uploadtime", "video_title"],
    )
    .to_pandas()
)

sents = dl.sentences(filtered=True).join(dl.popbert(filtered=True), "sentence_id").to_pandas()

sents = sents.groupby("video_id", observed=True).agg(
    n_sentences=("video_id", "size"),
    elite=("elite", "mean"),
    pplcentr=("pplcentr", "mean"),
)

videos = videos.merge(sents, on="video_id")

# Most Liked

In [5]:
top_like_videos = videos[videos.channel != "FDP"]

quantiles = top_like_videos.groupby("channel").video_likes.quantile(q=0.99).rename("quantile")

In [6]:
df = top_like_videos.merge(quantiles, how="left", on="channel")
df["top_1p"] = df.apply(lambda x: 1 if x.video_likes > x["quantile"] else 0, axis=1)

In [7]:
top_videos = (
    df.sort_values(["channel", "video_likes"], ascending=False)
    .groupby("channel")
    .head(
        10,
    )
    .set_index(["channel", "video_id"])
)

In [8]:
top_videos

video_likes  video_views video_uploadtime  \
channel video_id                                                 
SPD     HxtUEy0aY_U         7048       261702       2019-05-24   
        J5_6XjAEKWo         3215       124091       2019-12-06   
        Y8JWqzIEkYg         2943       562534       2021-08-04   
        R-Td0654RxU         1852       116863       2021-05-11   
        qcear4-NAG8         1200        38404       2020-12-12   
        4K5BGB34i_k          711       172277       2021-09-21   
        84DhwYfXY7c          619        19731       2021-12-31   
        J7WJ8UrUwJM          607        10036       2021-05-09   
        jvrswEl5WRY          500        49390       2023-02-08   
        aZx04PRQUb8          426         8247       2019-11-16   
Left    xJ2_ykd3qQg        19970       769998       2018-02-22   
        thsWrPlTCHU        15777       663977       2018-11-30   
        35yCb1A9bmc         6428       230089       2019-11-07   
        j_AS5WOaylM         4402       149768       2021-05-10   
        Vdtfq7uwrjE         3469       312521       2021-08-28   
        1gGx4mIOwtQ         3117       137967       2019-12-05   
        L3BWY8CWjjE         2396       303424       2021-09-12   
        j-GoyDi0NUA         1702       208320       2019-04-22   
        l5abgjJlcE4         1692        37786       2023-05-09   
        p41mETuQQ-E         1496        39602       2021-02-27   
Greens  Em_WUdK5WKI        12171       970937       2021-08-24   
        JVK9a9YAG1A          801        14152       2021-09-08   
        8onMba916cw          709        43383       2022-10-16   
        RUew3wECQBU          555        18255       2021-06-12   
        ddut9jiggeA          501        38978       2021-09-12   
        rMThbJVcY-Y          443         8264       2021-09-17   
        smOJuUd4yVA          405        18552       2022-10-14   
        5YjaQnY4BP4          394         8118       2021-10-29   
        vIGIkZhS258          376         7674       2021-09-19   
        3LSpC_z1m_k          373        18603       2020-11-20   
CSU     G5CBslnIYbs          429      1374514       2018-09-24   
        qV8plOy7mO8          273      1180669       2023-09-10   
        toVAryo-KIA          216         4520       2019-09-30   
        K0HX5OUY1cQ          172         3809       2019-11-20   
        pDyf3tu6_GQ          144         8304       2018-09-15   
        RKUU8D_Louc          102         3589       2020-09-11   
        KuUwqsw86Wk          100         7053       2021-02-17   
        -YMkFuGjPr4           99         4187       2021-08-31   
        DryIeBFXz00           97        20314       2019-04-18   
        IHmPYYkwsVE           96         6316       2019-10-17   
CDU     W71tUlrMeD4         3272      1366292       2021-08-22   
        kRgig-3p8J4         2672       180064       2023-07-27   
        TzGjX2K5na4         2454          946       2021-08-30   
        LrHKojPL7uU         2275         1441       2021-09-05   
        TtmRKq_k4_k         1557        95204       2021-09-18   
        QYfn1To3S0g         1444       212612       2021-01-16   
        MmnbRfL87yE          757        69953       2023-02-16   
        gt4XjbdkV6k          666        53562       2023-04-26   
        CrOVe9BhHAM          655       271192       2021-09-20   
        B59WbE0OvGA          606        38902       2023-12-22   
AfD TV  9-KekzU3AJU       114061      2708077       2023-06-22   
        qpGQ-l5o5Js        42502       492150       2021-11-24   
        N0R_bAITYBA        34632       494284       2023-11-28   
        FqZ8eKFJZU4        31729       401182       2022-03-07   
        sVgZHsnWqrE        29602       606412       2022-11-23   
        W-gBgUkLHAI        27685       328410       2021-12-15   
        S4-ZdKHSAr8        26691       644321       2019-05-04   
        UwAlL_3QU18        26557       337452       2023-06-06   
        1pksqcM2TBI        25903       490639       2022-02-1

In [9]:
top_videos.reset_index()[["channel", "video_likes", "video_views", "video_title"]].to_csv(
    src.OUT / "tables/most_liked_videos_per_channel.csv",
    index=False,
)

# Most Anti-Elitism

In [10]:
quantiles = videos.groupby("channel").elite.quantile(q=0.99).rename("quantile")

In [11]:
df = videos.merge(quantiles, how="left", on="channel")
df["top_1p"] = df.apply(lambda x: 1 if x.elite > x["quantile"] else 0, axis=1)

In [12]:
top_videos = (
    df.sort_values(["channel", "elite"], ascending=False)
    .groupby("channel")
    .head(
        10,
    )
    .set_index(["channel", "video_id"])
)

In [13]:
top_videos

video_likes  video_views video_uploadtime  \
channel video_id                                                 
SPD     F0c4_NPpnGA          142         2480       2024-01-18   
        d6vy8DvlA6k          177         2811       2020-03-05   
        2eJPpGsTI0c          180         2606       2021-08-30   
        2rwXzpbyDEU          165         3675       2020-03-05   
        y1WnueSc9fQ           61          839       2021-05-18   
        3HwEQBSzMUI           32          967       2023-12-09   
        B4hGX4jIMks          198         4516       2024-01-12   
        CtIAZCMJyWg          144         5122       2022-03-25   
        nuVHZu54xvU          111         2109       2021-09-18   
        67wvrzYddRo           77          493       2021-08-14   
Left    l5abgjJlcE4         1692        37786       2023-05-09   
        MUgDgKGu5Bg          431         7201       2021-09-02   
        9DC619q6mVA           24          630       2019-04-05   
        FFg9z4i9lqM          119         2704       2022-12-06   
        35yCb1A9bmc         6428       230089       2019-11-07   
        GSfgp8r9h3A           91         2762       2018-06-18   
        s15ADZa3BvI          177         2270       2023-09-21   
        L3wllg-iZNg          109         1488       2022-11-25   
        zN5YQIN5ep8          168         3614       2021-09-20   
        quIkOTiaCSA           75          989       2018-09-26   
Greens  BKA5Rs9r7WE            5          281       2018-01-27   
        iM80OK5_e9E           13          498       2018-01-27   
        sTFRFy4jcN4           22         1868       2018-11-10   
        fvgOxVAw5hQ            6          212       2018-01-27   
        kMlQJQjV__M          194         6922       2019-09-20   
        5aAfx-1usQ0           84         1113       2018-06-29   
        neJCKmYZZBc           42         1133       2019-11-16   
        QO_Y14cx8Lo           51          774       2021-06-13   
        _cfV76TxsY8           15         1722       2019-11-16   
        YjN_Q4JIQjw           41         2924       2022-10-14   
FDP     UYtJ1msQuYM            0          547       2019-12-02   
        pdlIAFuKRQI            0         2598       2020-02-14   
        4yJCimj2O_U            0          564       2018-01-08   
        exvX1gSRtD0            0          567       2019-12-17   
        TbTJccB-Up8            0         2333       2020-03-02   
        QICTmpFHwmQ            0          812       2019-07-04   
        Ie6tihXg7A4            0          825       2019-12-16   
        6n0TgxBjWuc            0        41236       2021-07-30   
        rDZdmV-4DgI            0         1493       2018-01-08   
        1ze5fpz3oFE            0         5300       2019-04-29   
CSU     toVAryo-KIA          216         4520       2019-09-30   
        nDYYxXPqXGQ           71         5213       2023-02-22   
        N431nIgfOnA           21          641       2020-02-05   
        K0HX5OUY1cQ          172         3809       2019-11-20   
        iPKZDFAdIYs           28          472       2019-10-05   
        983lQr9XElg           82         2932       2021-09-16   
        16jEAhKmF-I           35          955       2021-09-25   
        6eSiwakBcgY           21          323       2021-09-20   
        pJVOIfs3h-k           29         2884       2019-03-31   
        -YMkFuGjPr4           99         4187       2021-08-31   
CDU     EksM0bVVH1M           36         1763       2020-02-05   
        bbcjzNNtveo           18          334       2022-01-19   
        xgkXD9s4Opc          543        20013       2020-02-26   
        oYAl63O10gU           15          693       2019-04-08   
        6_RMW7sF4kM           69         2893       2023-06-20   
        t9oeOo08rGM            8          190       2022-01-24   
        q64peGPwHV8           46          965       2021-09-22   
        GfIbnaLOS7M          174         4009       2021-09-08   
        mINqf00pBts           18          569       2019-09-1

In [14]:
top_videos.reset_index()[["channel", "n_sentences", "elite", "video_title"]].to_csv(
    src.OUT / "tables/most_antielitism_videos_per_channel.csv",
    index=False,
)